# 🤖 AutoML Pilot - Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset. This template is designed to work seamlessly with datasets exported from the AutoML Pilot web app.

In [ ]:
# @title 1. Install Dependencies
# Install PyCaret and other necessary libraries
!pip install -q pycaret[full] ydata-profiling pandas
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Load Dataset
import pandas as pd
import os
from google.colab import files

print('📤 Upload your dataset.csv file (or use the path below if already in Drive/Local)')
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)
    print(f'✅ Loaded uploaded dataset: {df.shape[0]} rows x {df.shape[1]} columns')
    display(df.head())
else:
    # Fallback to path if no file uploaded
    dataset_path = '/content/dataset.csv' # @param {type:"string"}
    if os.path.exists(dataset_path):
        df = pd.read_csv(dataset_path)
        print(f'✅ Loaded dataset from path: {df.shape[0]} rows x {df.shape[1]} columns')
        display(df.head())
    else:
        print(f'❌ Error: No file uploaded or found at {dataset_path}')

In [ ]:
# @title 3. Automated EDA
from ydata_profiling import ProfileReport

print('📊 Generating EDA Report...')
profile = ProfileReport(df, title="Dataset Profiling Report", explorative=True)
profile.to_notebook_iframe()

In [ ]:
# @title 4. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize

# Configuration
target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]

print(f'🚀 Starting AutoML for {task_type} on target: {target_column}')

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False)
    print('Comparing models...')
    best_model = cls_compare()
    leaderboard = cls_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_classification_model')
    print('✅ Classification model saved as best_classification_model.pkl')

else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False)
    print('Comparing models...')
    best_model = reg_compare()
    leaderboard = reg_pull()
    display(leaderboard)
    
    # Finalize and Save
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_regression_model')
    print('✅ Regression model saved as best_regression_model.pkl')

In [ ]:
# @title 5. Email Results
import smtplib, ssl, json
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

send_email = True # @param {type:"boolean"}
user_email = "" # @param {type:"string"}
smtp_user = "" # @param {type:"string"}
smtp_pass = "" # @param {type:"string"}

if send_email and user_email and smtp_user and smtp_pass:
    print(f'📧 Sending results to {user_email}...')
    try:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = f"AutoML Pilot Colab Results - {task_type.capitalize()}"
        msg["From"] = smtp_user
        msg["To"] = user_email
        
        # Simple HTML table for leaderboard
        html_table = leaderboard.to_html(classes='table table-striped')
        
        html = f"""
        <html>
          <body>
            <h2>🚀 AutoML Pilot Training Report</h2>
            <p>Your model training in Colab is complete!</p>
            <h3>📊 Leaderboard</h3>
            {html_table}
            <p>Download the .pkl file from Colab and upload it to the AutoML Pilot 'Deployment' tab.</p>
          </body>
        </html>
        """
        msg.attach(MIMEText(html, "html"))
        
        context = ssl.create_default_context()
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(smtp_user, smtp_pass)
            server.sendmail(smtp_user, user_email, msg.as_string())
        print('✅ Email sent successfully!')
    except Exception as e:
        print(f'❌ Failed to send email: {e}')
else:
    print('ℹ️ Emailing skipped. Configure credentials to enable.')

In [ ]:
# @title 6. Download Model
from google.colab import files

model_name = 'best_classification_model.pkl' if task_type == 'classification' else 'best_regression_model.pkl'
if os.path.exists(model_name):
    files.download(model_name)
    print(f'✅ Triggered download for {model_name}')
    print('💡 **Next Step:** Go to the AutoML Pilot website, open the **Deployment** tab, and upload this file.')
else:
    alt_name = model_name.replace('.pkl', '')
    if os.path.exists(alt_name + '.pkl'):
        files.download(alt_name + '.pkl')
        print(f'✅ Triggered download for {alt_name}.pkl')
    else:
        print('❌ Model file not found. Ensure training completed successfully.')